# 11 - Transformers

**AI sin humo** - Notas personales para entender deep learning desde cero.

El Transformer (Vaswani et al., 2017 - "Attention is All You Need") eliminó las RNNs por completo. Usa **solo attention** (con algunas cosas extra) para procesar secuencias. Es la base de GPT, BERT, y todos los LLMs modernos.

---

## Contenido

1. [Multi-Head Attention](#multi-head)
2. [Scaled dot-product attention](#scaled)
3. [Causal attention (masked)](#causal)
4. [Transformer block completo](#block)
5. [Input: embeddings + positional encoding](#input)
6. [Arquitectura original (encoder-decoder)](#original)
7. [GPT architecture (decoder-only)](#gpt)
8. [Múltiples capas: cómo se refina el contexto](#capas)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

---

<a id='multi-head'></a>
## 1. Multi-Head Attention

En el notebook anterior vimos self-attention con un solo set de Q, K, V. Pero en la práctica se usa **Multi-Head Attention**: se hacen **N atenciones en paralelo**, cada una con sus propias matrices W_Q, W_K, W_V.

![Attention block](../ai_notas/AI%20notas/image%2058.png)

### ¿Por qué múltiples heads?

Cada head puede capturar **aspectos diferentes** de las relaciones entre tokens:
- Un head puede atender a relaciones sintácticas (sujeto-verbo)
- Otro a relaciones semánticas (pronombre-referente)
- Otro a posición relativa

### ¿Cómo funciona?

Si `d_model` es la dimensión del modelo (ej: 512) y tenemos `n_heads` heads (ej: 8):

1. **Cada head** tiene sus propias matrices W_Q, W_K, W_V de tamaño `(d_model, d_head)` donde `d_head = d_model / n_heads` (ej: 64)
2. Cada head hace attention independientemente, produciendo un output de `(seq_len, d_head)`
3. Se **concatenan** todos los outputs: `(seq_len, d_head * n_heads) = (seq_len, d_model)`
4. Se pasa por una **proyección lineal** final: `W_O` de tamaño `(d_model, d_model)`

En la práctica, las N matrices chiquitas se pueden implementar como una sola grande y después reshapear.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention."""
    
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        
        # One big matrix for all heads (more efficient than N small ones)
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        B, T, C = x.shape  # batch, seq_len, d_model
        
        # Project to Q, K, V for all heads at once
        qkv = self.W_qkv(x)  # (B, T, 3*d_model)
        q, k, v = qkv.chunk(3, dim=-1)  # each (B, T, d_model)
        
        # Reshape to (B, n_heads, T, d_head)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        
        # Scaled dot-product attention per head
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)  # (B, n_heads, T, T)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Weighted sum of values
        context = attn_weights @ v  # (B, n_heads, T, d_head)
        
        # Concat heads: (B, T, d_model)
        context = context.transpose(1, 2).contiguous().view(B, T, self.d_model)
        
        # Final projection
        output = self.W_o(context)
        return output


# Quick test
mha = MultiHeadAttention(d_model=32, n_heads=4)
x = torch.randn(2, 10, 32)  # batch=2, seq_len=10, d_model=32
out = mha(x)
print(f"Input: {x.shape} → Output: {out.shape}")
print(f"Cada head opera en d_head = {mha.d_head}")

---

<a id='scaled'></a>
## 2. Scaled dot-product attention

El "scaled" viene de dividir por $\sqrt{d_k}$:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

### ¿Por qué escalar?

Si no escalamos, cuando $d_k$ es grande, los dot products tienden a ser grandes en magnitud. El softmax de valores grandes se vuelve **muy peaked** (casi one-hot), lo que genera gradientes muy chicos y dificulta el entrenamiento.

Al dividir por $\sqrt{d_k}$, mantenemos la varianza del dot product en ~1, independientemente de la dimensión.

---

<a id='causal'></a>
## 3. Causal attention (masked)

En un **decoder** (como GPT), el modelo genera tokens **de izquierda a derecha**. No puede mirar tokens futuros porque aún no fueron generados.

Para esto usamos una **máscara causal**: una matriz triangular inferior que bloquea la atención a posiciones futuras.

```
Token 0 puede ver: [0]
Token 1 puede ver: [0, 1]
Token 2 puede ver: [0, 1, 2]
Token 3 puede ver: [0, 1, 2, 3]
```

Se implementa poniendo $-\infty$ en las posiciones superiores de la matriz de scores **antes** del softmax. Así esas posiciones quedan con probabilidad 0.

In [ ]:
# Causal mask visualization
T = 6
causal_mask = torch.tril(torch.ones(T, T))  # lower triangular

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# The mask (1 = can attend, 0 = blocked)
axes[0].imshow(causal_mask, cmap='Blues')
axes[0].set_title('Máscara causal\n(1=puede ver, 0=bloqueado)')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')

# Scores before masking (random)
torch.manual_seed(42)
scores = torch.randn(T, T)
axes[1].imshow(scores, cmap='RdBu')
axes[1].set_title('Scores antes de máscara')

# After masking: -inf where mask=0
masked_scores = scores.masked_fill(causal_mask == 0, float('-inf'))
axes[2].imshow(F.softmax(masked_scores, dim=-1), cmap='Blues')
axes[2].set_title('Pesos de atención (post-softmax)\nFuturo = 0')

plt.tight_layout()
plt.show()

---

<a id='block'></a>
## 4. Transformer block completo

![Transformer block](../ai_notas/AI%20notas/image%2057.png)

Un **transformer block** tiene esta estructura:

```
x → LayerNorm → Multi-Head Attention → + (residual con x) → 
  → LayerNorm → FeedForward → + (residual) → output
```

### FeedForward

Después del attention, cada token pasa por una red feedforward **individual** (misma red para todos los tokens). Esto sirve para transformar la info que ya tiene cada token de forma no contextual.

Típicamente: `Linear(d_model, 4*d_model) → GELU → Linear(4*d_model, d_model)`

### Add & Norm

Residual connection (sumar el input original) + LayerNorm. Esto estabiliza el entrenamiento y permite apilar muchas capas.

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feedforward: expand 4x, GELU, project back."""
    def __init__(self, d_model, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """One transformer block: attention + feedforward with residuals and LayerNorm."""
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, dropout)
    
    def forward(self, x, mask=None):
        # Pre-norm: LayerNorm before attention (GPT-style)
        x = x + self.attn(self.ln1(x), mask)   # residual
        x = x + self.ffn(self.ln2(x))           # residual
        return x


# Test
block = TransformerBlock(d_model=32, n_heads=4)
x = torch.randn(2, 10, 32)
out = block(x)
print(f"Input: {x.shape} → Output: {out.shape}")
print(f"Mismo shape: la info se transforma pero no cambia de tamaño.")

---

<a id='input'></a>
## 5. Input: embeddings + positional encoding

Los tokens de texto se convierten en números (índices del vocabulario). Esos índices se pasan por una **embedding table** que asigna a cada token un vector de `d_model` dimensiones. Estos embeddings son **aprendibles** durante el entrenamiento.

Pero attention es **permutation-invariant**: no le importa el orden. "El gato come" y "Come el gato" darían lo mismo sin información posicional.

Para resolver esto, se le **suma** un **positional embedding** a cada token. Puede ser:
- **Aprendible**: una tabla más, de tamaño `(max_seq_len, d_model)`
- **Fijo (sinusoidal)**: funciones seno y coseno de distintas frecuencias (el original de Vaswani)

$$\text{Input} = \text{TokenEmbedding}(\text{token\_id}) + \text{PositionalEmbedding}(\text{posición})$$

In [ ]:
# Positional encoding: sinusoidal (original Vaswani)
def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


pe = sinusoidal_positional_encoding(50, 32)
fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(pe.T, cmap='RdBu', aspect='auto')
ax.set_xlabel('Posición en la secuencia')
ax.set_ylabel('Dimensión del embedding')
ax.set_title('Positional Encoding sinusoidal\n(cada posición tiene un patrón único)')
plt.tight_layout()
plt.show()

---

<a id='original'></a>
## 6. Arquitectura original (encoder-decoder)

![Transformer original](../ai_notas/AI%20notas/image%2059.png)

El Transformer original de Vaswani tiene **dos partes**:

### Encoder
- Procesa la secuencia de **input** (ej: oración en inglés para traducir)
- Self-attention **bidireccional**: cada token puede ver TODOS los demás
- Múltiples capas de transformer blocks
- Genera representaciones ricas para cada token

### Decoder
- Genera la secuencia de **output** token a token (ej: traducción al español)
- **Causal self-attention** (masked): solo puede ver tokens anteriores
- **Cross-attention**: además de mirarse a sí mismo, el decoder puede "mirar" al encoder. Q viene del decoder, K y V vienen del encoder.
- Múltiples capas

### Cross-attention

Es como self-attention pero las queries vienen del decoder y las keys/values del encoder. Así el decoder puede "buscar" información relevante del input para generar cada token del output.

---

<a id='gpt'></a>
## 7. GPT architecture (decoder-only)

![GPT architecture](../ai_notas/AI%20notas/image%2060.png)

GPT simplifica el Transformer original: **solo usa el decoder**, sin encoder ni cross-attention.

```
Token indices → Token Embedding + Positional Embedding
    → Transformer Block 1 (causal self-attention + FFN)
    → Transformer Block 2
    → ...
    → Transformer Block N
    → LayerNorm
    → Linear (d_model → vocab_size)
    → Softmax → Probabilidades sobre el vocabulario
```

Para cada posición en la secuencia, el modelo predice el **siguiente token**. Se entrena con cross-entropy comparando la predicción con el token real.

In [ ]:
class GPT(nn.Module):
    """Minimal GPT: decoder-only transformer."""
    
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_seq_len, dropout=0.0):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)  # learned positional
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, dropout) for _ in range(n_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.max_seq_len = max_seq_len
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        # Embeddings
        tok_emb = self.token_emb(idx)  # (B, T, d_model)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))  # (T, d_model)
        x = tok_emb + pos_emb  # (B, T, d_model)
        
        # Causal mask
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0).unsqueeze(0)
        
        # Transformer blocks
        for block in self.blocks:
            x = block(x, mask)
        
        x = self.ln_final(x)  # (B, T, d_model)
        logits = self.head(x)  # (B, T, vocab_size)
        
        # Loss
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Autoregressive generation."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]  # crop to max context
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # last position
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx


# Quick test
model = GPT(vocab_size=256, d_model=64, n_heads=4, n_layers=2, max_seq_len=32)
idx = torch.randint(0, 256, (1, 10))
logits, _ = model(idx)
print(f"Input: {idx.shape} → Logits: {logits.shape}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

---

<a id='capas'></a>
## 8. Múltiples capas: cómo se refina el contexto

Hay múltiples iteraciones de attention. Tiene una especie de jerarquización al pasar por múltiples capas, como en todas las redes neuronales.

- **Capa 0**: Los inputs son embeddings independientes por token (+ positional). Q, K, V se proyectan de ahí. Attention combina tokens: para un token $i$, su Q busca en todos los K, pondera V. El output ya tiene algo de contexto básico.

- **Capa 1**: Attention opera sobre representaciones ya mezcladas de la capa 0. Más contexto.

- **Capa 2+**: Cada capa toma el output de la anterior y lo enriquece más. Es como una "recurrencia implícita" sin vanishing gradients.

**Intuitivo**: Capa 1 captura relaciones locales (sujeto-verbo). Capa 2 conecta frases distantes (pronombre al referente). Al final (6-12 capas típicas), cada token lleva contexto profundo de toda la secuencia.

### Diferencia clave con RNNs

- RNN: secuencial, un token a la vez, vanishing gradients
- Transformer: **paralelo**, todos los tokens al mismo tiempo, acceso directo a cualquier posición

Esto hace que los transformers sean **mucho más rápidos** de entrenar (aprovechan GPUs) y **mucho mejores** capturando dependencias de largo alcance.

---

## Resumen

| Concepto | Descripción |
|----------|-------------|
| **Multi-Head Attention** | N atenciones en paralelo, cada head captura aspectos diferentes. Se concatenan y proyectan. |
| **Scaled** | Dividir por $\sqrt{d_k}$ para mantener gradientes estables en softmax. |
| **Causal mask** | Máscara triangular que impide ver tokens futuros (decoder). |
| **Transformer block** | LayerNorm → MHA → residual → LayerNorm → FFN → residual |
| **Positional encoding** | Suma de embeddings que codifican la posición (aprendible o sinusoidal). |
| **Encoder-decoder** | Encoder bidireccional + decoder causal con cross-attention (traducción). |
| **GPT (decoder-only)** | Solo causal self-attention. Predice el siguiente token. Base de los LLMs. |
| **Múltiples capas** | Cada capa refina el contexto. Recurrencia implícita sin vanishing gradients. |

---

**Siguiente notebook →** [12 - GPT from Scratch](./12_gpt_from_scratch.ipynb): implementar y entrenar un GPT completo.